# VibeMol in JupyterLab

Serve the checkout from the repository root with `python3 -m http.server 8000 --bind 127.0.0.1`.
Install the widget with `pip install -e python`, then run these cells one at a time.
Set `VIBEMOL_VIEWER_URL` to use another VibeMol server that includes the embed bridge.

In [ ]:
import os
from pathlib import Path

from IPython.display import display
from vibemol import Viewer

viewer = Viewer(url=os.environ.get("VIBEMOL_VIEWER_URL"), source="local")
display(viewer)

In [ ]:
loading = viewer.load(Path("../tests/data/cyclopropene_cas43.molden"))

Once cyclopropene is visible, run the next cell to read the load receipt. Browser replies reach the kernel asynchronously.

In [ ]:
print(viewer.ready, viewer.app_info["appVersion"])
print(loading.result()["loadedCount"], "Molden file loaded")

Switch the viewer to Edit mode and activate the Transform tool.
Click one carbon and **Shift-click** a second carbon to extend the selection.
The Edit viewport uses Shift for additive selection. Ctrl-click is a camera orbit gesture.
If atoms overlap, orbit first to separate them on screen. Run the next cell after both atoms are highlighted.

In [ ]:
print(viewer.selection)
basis = viewer.run("calculations.ready")

**Shift-click the third carbon** to define the π plane. The export needs three non-collinear atoms.
The next cell assigns the whole 2p shell on each carbon and sets the AVAS options.

In [ ]:
basis.result()
selection = viewer.selection
assert selection and set(selection["indices"]) == {0, 1, 2}, "Select all three carbon atoms"
assignments = {row["id"]: row["shells"] for row in viewer.calculations["selections"]}
shell_jobs = [
    viewer.run("calculations.selectShell", index=index, shell="2p")
    for index, atom_id in zip(selection["indices"], selection["atomIds"])
    if assignments.get(atom_id, {}).get("2p") != "all"
]
options = viewer.run("calculations.configure", method="cumulative", sigma=0.98)

In [ ]:
for job in shell_jobs:
    job.result()
options.result()
plane = {"kind": "piPlane", "seed": {"kind": "ids", "ids": selection["atomIds"]}}
exported = viewer.run("forte2.export", avas={"planes": [plane]})

Read the AVAS arguments in a later cell. VibeMol prepares the forte2 input. The notebook does not run a calculation here.

In [ ]:
exported.result()["avas"]["arguments"]